In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Cell 1 - Load data from warehouse tables
import mlflow
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import when, col

# Load payments and tenants
df_payments = spark.table("payments_clean")
df_tenants = spark.table("tenants_clean")

# Join
df = df_payments.join(df_tenants, on="tenant_id", how="inner")

# Create binary label: 1 = late/failed, 0 = on time
df = df.withColumn("label", when(col("payment_status").isin("Late", "Failed"), 1).otherwise(0))

print("Total records:", df.count())
print("Late/Failed:", df.filter(col("label") == 1).count())
print("On Time:", df.filter(col("label") == 0).count())

StatementMeta(, 26ca8a04-0e56-4c2b-9d97-1a34b2cb0d25, 3, Finished, Available, Finished, False)

Total records: 4272
Late/Failed: 688
On Time: 3584


In [2]:
# Cell 2 - Prepare features and train model

# Select features
feature_cols = ["credit_score", "annual_income", "monthly_rent", "days_late", "lease_term_months"]

# Drop nulls
df_model = df.select(feature_cols + ["label"]).dropna()

# Assemble features
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Train/test split
train, test = df_model.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count()} | Test: {test.count()}")

# Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, maxDepth=5, seed=42)

# Pipeline
pipeline = Pipeline(stages=[assembler, rf])

# MLflow tracking
mlflow.set_experiment("Late_Payment_Prediction")

with mlflow.start_run(run_name="RandomForest_v1"):
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_param("features", feature_cols)
    
    # Train
    model = pipeline.fit(train)
    
    # Predict
    predictions = model.transform(test)
    
    # Evaluate
    auc_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
    acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    f1_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
    
    auc = auc_eval.evaluate(predictions)
    acc = acc_eval.evaluate(predictions)
    f1 = f1_eval.evaluate(predictions)
    
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
    
    print(f"\nModel Results:")
    print(f"AUC:      {auc:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    # Feature importance
    rf_model = model.stages[-1]
    importances = list(zip(feature_cols, rf_model.featureImportances.toArray()))
    importances.sort(key=lambda x: x[1], reverse=True)
    print(f"\nFeature Importances:")
    for feat, imp in importances:
        print(f"  {feat}: {imp:.4f}")
    
    mlflow.log_param("top_feature", importances[0][0])

StatementMeta(, 26ca8a04-0e56-4c2b-9d97-1a34b2cb0d25, 4, Finished, Available, Finished, False)

Train: 3467 | Test: 805

Model Results:
AUC:      1.0000
Accuracy: 1.0000
F1 Score: 1.0000

Feature Importances:
  days_late: 0.9320
  credit_score: 0.0650
  annual_income: 0.0016
  monthly_rent: 0.0008
  lease_term_months: 0.0006


2026/04/07 16:29:25 INFO mlflow.tracking.fluent: Experiment with name 'Late_Payment_Prediction' does not exist. Creating a new experiment.


In [3]:
# Cell 3 - Retrain without data leakage (remove days_late)

feature_cols_v2 = ["credit_score", "annual_income", "monthly_rent", "lease_term_months"]

df_model_v2 = df.select(feature_cols_v2 + ["label"]).dropna()

train2, test2 = df_model_v2.randomSplit([0.8, 0.2], seed=42)

assembler2 = VectorAssembler(inputCols=feature_cols_v2, outputCol="features")
rf2 = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, maxDepth=5, seed=42)
pipeline2 = Pipeline(stages=[assembler2, rf2])

mlflow.set_experiment("Late_Payment_Prediction")

with mlflow.start_run(run_name="RandomForest_v2_noleak"):
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_param("features", feature_cols_v2)
    mlflow.log_param("note", "Removed days_late to prevent leakage")
    
    model2 = pipeline2.fit(train2)
    predictions2 = model2.transform(test2)
    
    auc2 = auc_eval.evaluate(predictions2)
    acc2 = acc_eval.evaluate(predictions2)
    f12 = f1_eval.evaluate(predictions2)
    
    mlflow.log_metric("auc", auc2)
    mlflow.log_metric("accuracy", acc2)
    mlflow.log_metric("f1_score", f12)
    
    rf_model2 = model2.stages[-1]
    importances2 = list(zip(feature_cols_v2, rf_model2.featureImportances.toArray()))
    importances2.sort(key=lambda x: x[1], reverse=True)
    
    print(f"Model v2 Results (no leakage):")
    print(f"AUC:      {auc2:.4f}")
    print(f"Accuracy: {acc2:.4f}")
    print(f"F1 Score: {f12:.4f}")
    print(f"\nFeature Importances:")
    for feat, imp in importances2:
        print(f"  {feat}: {imp:.4f}")

StatementMeta(, 26ca8a04-0e56-4c2b-9d97-1a34b2cb0d25, 5, Finished, Available, Finished, False)

Model v2 Results (no leakage):
AUC:      0.8527
Accuracy: 0.8522
F1 Score: 0.7990

Feature Importances:
  credit_score: 0.8463
  annual_income: 0.0700
  monthly_rent: 0.0561
  lease_term_months: 0.0276
